## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Load the Data

In [2]:
data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

In [3]:
print("Shape:", X.shape)
print("Target:", data.target_names)
print(y.describe())

Shape: (20640, 8)
Target: ['MedHouseVal']
count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64


## Train Ready

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Diff Exp Models

In [5]:
lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.5757877060324514
MSE: 0.5558915986952435


In [6]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.774774131273305
MSE: 0.29513800051156747


In [7]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.8408716044998452
MSE: 0.2085232781564716


## Prepare for ML Flow

In [8]:
models = [

    (
        "Linear Regression",
        LinearRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBoost",
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    )

]

In [9]:
reports = []

for model_name, model, train_set, test_set in models:

    X_tr, y_tr = train_set
    X_te, y_te = test_set

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_te)

    report = {
        "mse": mean_squared_error(y_te, predictions),
        "rmse": float(np.sqrt(mean_squared_error(y_te, predictions))),
        "mae": mean_absolute_error(y_te, predictions),
        "r2": r2_score(y_te, predictions)
    }

    reports.append(report)

In [10]:
reports

[{'mse': 0.5558915986952435,
  'rmse': 0.7455813830127758,
  'mae': 0.5332001304956557,
  'r2': 0.5757877060324514},
 {'mse': 0.29513800051156747,
  'rmse': 0.5432660494744426,
  'mae': 0.3657301973669837,
  'r2': 0.774774131273305},
 {'mse': 0.2085232781564716,
  'rmse': 0.45664349131075066,
  'mae': 0.2959191474993146,
  'r2': 0.8408716044998452}]

## Exp Tracking ML Flow Local

In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("California Housing Regression PBLM 1")

2026/08/06 11:45:54 INFO mlflow.tracking.fluent: Experiment with name 'California Housing Regression PBLM 1' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/HP/DevOps/MLflow-101-Regression/mlruns/1', creation_time=1785996954081, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785996954081, lifecycle_stage='active', name='California Housing Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

In [12]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        #Params
        mlflow.log_param("Model", model_name)

        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())

        #Mts - R2 is the main metric we care about here
        mlflow.log_metric("R2", report["r2"])
        mlflow.log_metric("MSE", report["mse"])
        mlflow.log_metric("RMSE", report["rmse"])
        mlflow.log_metric("MAE", report["mae"])

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(
                model,
                name="model"
            )
        else:
            mlflow.sklearn.log_model(
                model,
                name="model"
            )

🏃 View run Linear Regression at: http://127.0.0.1:5000/#/experiments/1/runs/5c14b27a337e4223a7f18eb32629722f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/1/runs/80560d43d40740048e54fc04711dd667
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/2775cf4920224b4ba52ed92cd6dcd413
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Pick the Best Model (by R2)

In [13]:
best_index = np.argmax(
    [report["r2"] for report in reports]
)

best_model_name, best_model, _, _ = models[best_index]

best_report = reports[best_index]

print("Model:", best_model_name)
print("R2:", best_report["r2"])
print("RMSE:", best_report["rmse"])

Model: XGBoost
R2: 0.8408716044998452
RMSE: 0.45664349131075066


In [14]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    #Params
    mlflow.log_param("Model", best_model_name)
    mlflow.log_param("Selection_Metric", "R2 Score")
    mlflow.log_param("Dataset", "California Housing")
    mlflow.log_params(best_model.get_params())

    #Metrics
    mlflow.log_metric("R2", best_report["r2"])
    mlflow.log_metric("MSE", best_report["mse"])
    mlflow.log_metric("RMSE", best_report["rmse"])
    mlflow.log_metric("MAE", best_report["mae"])

    mlflow.set_tag("Model_Type", best_model_name)
    mlflow.set_tag("Stage", "Candidate")
    mlflow.set_tag("Task", "Regression")

    if "XGBoost" in best_model_name:
        model_info = mlflow.xgboost.log_model(
            best_model,
            name="model",
            registered_model_name="California_Housing_Best_Model"
        )
    else:
        model_info = mlflow.sklearn.log_model(
            best_model,
            name="model",
            registered_model_name="California_Housing_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID       :", run_id)
print("Model URI    :", model_uri)
print("Model Name   :", "California_Housing_Best_Model")

Successfully registered model 'California_Housing_Best_Model'.
2026/08/06 11:46:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: California_Housing_Best_Model, version 1


🏃 View run Champion_XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/31308f01b3174716aeafc5fd3d10fc56
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID       : 31308f01b3174716aeafc5fd3d10fc56
Model URI    : models:/m-9d44c600025b41eea4977d9a89769501
Model Name   : California_Housing_Best_Model


Created version '1' of model 'California_Housing_Best_Model'.


## Loading and Pushing to Prod

In [15]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

latest_version = client.get_latest_versions(
    "California_Housing_Best_Model"
)[0]

print("Model Name:", latest_version.name)
print("Version:", latest_version.version)
print("Stage:", latest_version.current_stage)

Model Name: California_Housing_Best_Model
Version: 1
Stage: None


In [16]:
model_uri = f"models:/California_Housing_Best_Model/{latest_version.version}"

model = mlflow.pyfunc.load_model(model_uri)

print("Model loaded successfully")

Model loaded successfully


In [17]:
predictions = model.predict(X_test)

print(predictions[:10])
print("Reload R2 check:", r2_score(y_test, predictions))

[0.5592586  0.88267744 5.2782254  2.510783   2.331231   1.534694
 2.211278   1.615608   2.665137   4.9998064 ]
Reload R2 check: 0.8408716044998452


In [18]:
client.update_model_version(
    name="California_Housing_Best_Model",
    version=latest_version.version,
    description="""
    Champion model for California Housing Regression.

    Tested successfully before production deployment.

    Dataset:
    sklearn California Housing

    Metric:
    R2 Score
    """
)

<ModelVersion: aliases=[], creation_timestamp=1785996984688, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('\n'
 '    Champion model for California Housing Regression.\n'
 '\n'
 '    Tested successfully before production deployment.\n'
 '\n'
 '    Dataset:\n'
 '    sklearn California Housing\n'
 '\n'
 '    Metric:\n'
 '    R2 Score\n'
 '    '), last_updated_timestamp=1785996984876, metrics=None, model_id=None, name='California_Housing_Best_Model', params=None, run_id='31308f01b3174716aeafc5fd3d10fc56', run_link='', source='models:/m-9d44c600025b41eea4977d9a89769501', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [19]:
client.transition_model_version_stage(
    name="California_Housing_Best_Model",
    version=latest_version.version,
    stage="Production"
)

<ModelVersion: aliases=[], creation_timestamp=1785996984688, current_stage='Production', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('\n'
 '    Champion model for California Housing Regression.\n'
 '\n'
 '    Tested successfully before production deployment.\n'
 '\n'
 '    Dataset:\n'
 '    sklearn California Housing\n'
 '\n'
 '    Metric:\n'
 '    R2 Score\n'
 '    '), last_updated_timestamp=1785996984912, metrics=None, model_id=None, name='California_Housing_Best_Model', params=None, run_id='31308f01b3174716aeafc5fd3d10fc56', run_link='', source='models:/m-9d44c600025b41eea4977d9a89769501', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [20]:
production_model = mlflow.pyfunc.load_model(
    "models:/California_Housing_Best_Model/Production"
)
prediction = production_model.predict(X_test)
print(prediction[:10])
print("Production R2:", r2_score(y_test, prediction))

[0.5592586  0.88267744 5.2782254  2.510783   2.331231   1.534694
 2.211278   1.615608   2.665137   4.9998064 ]


Production R2: 0.8408716044998452
